In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import torch
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import time

In [2]:
# CONSTANT
CLOCK_RAW_BASE_DIR = ""
COUNTER_RAW_BASE_DIR = ""
# merged datasets
OUTPUT_BASE_DIR = ""
CIRCUIT_LIST = [
    "simple_circuit",
    "montreal_circuit",
    "montmelo_circuit",
    "nurburgring_circuit"
]
TEST_DATASET = 'simple_circuit'

In [3]:
TRAIN_DATASET_NAME = 'train.csv'

## load clockwise dataset

In [4]:
# simple circuit
clock_train_df_simple_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[0],TRAIN_DATASET_NAME))
clock_test_df_simple_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[0],"test.csv"))

# montreal circuit
clock_train_df_montreal_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[1],TRAIN_DATASET_NAME))
clock_test_df_montreal_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[1],"test.csv"))

# montmelo circuit
clock_train_df_montmelo_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[2],TRAIN_DATASET_NAME))
clock_test_df_montmelo_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[2],"test.csv"))

# nurburgring circuit
clock_train_df_nurburgring_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[3],TRAIN_DATASET_NAME))
clock_test_df_nurburgring_circuit = pd.read_csv(os.path.join(CLOCK_RAW_BASE_DIR,CIRCUIT_LIST[3],"test.csv"))

In [5]:
clock_train_datasets_list = {
    "simple_circuit": clock_train_df_simple_circuit,
    "montreal_circuit": clock_train_df_montreal_circuit,
    "montmelo_circuit": clock_train_df_montmelo_circuit,
    "nurburgring_circuit": clock_train_df_nurburgring_circuit
}
clock_test_datasets_list = {
    "simple_circuit": clock_test_df_simple_circuit,
    "montreal_circuit": clock_test_df_montreal_circuit,
    "montmelo_circuit": clock_test_df_montmelo_circuit,
    "nurburgring_circuit": clock_test_df_nurburgring_circuit
}

## load counter clockwise dataset

In [6]:
# simple circuit
counter_train_df_simple_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[0],TRAIN_DATASET_NAME))
counter_test_df_simple_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[0],"test.csv"))

# montreal circuit
counter_train_df_montreal_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[1],TRAIN_DATASET_NAME))
counter_test_df_montreal_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[1],"test.csv"))

# montmelo circuit
counter_train_df_montmelo_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[2],TRAIN_DATASET_NAME))
counter_test_df_montmelo_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[2],"test.csv"))

# nurburgring circuit
counter_train_df_nurburgring_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[3],TRAIN_DATASET_NAME))
counter_test_df_nurburgring_circuit = pd.read_csv(os.path.join(COUNTER_RAW_BASE_DIR,CIRCUIT_LIST[3],"test.csv"))

In [7]:
counter_train_datasets_list = {
    "simple_circuit": counter_train_df_simple_circuit,
    "montreal_circuit": counter_train_df_montreal_circuit,
    "montmelo_circuit": counter_train_df_montmelo_circuit,
    "nurburgring_circuit": counter_train_df_nurburgring_circuit
}
counter_test_datasets_list = {
    "simple_circuit": counter_test_df_simple_circuit,
    "montreal_circuit": counter_test_df_montreal_circuit,
    "montmelo_circuit": counter_test_df_montmelo_circuit,
    "nurburgring_circuit": counter_test_df_nurburgring_circuit
}

In [8]:
start_time = time.time()

for test_circuit in CIRCUIT_LIST:

    # 1. CREATE CSV FILE
    test_dataset = test_circuit
    train_dataset_list = []
    test_dataset_list = []

    # take clock and counter clockwise train-test datasets from each circuit
    #train merge
    train_dataset_list.append(clock_train_datasets_list[test_dataset])
    train_dataset_list.append(counter_train_datasets_list[test_dataset])

    #test merge
    test_dataset_list.append(clock_test_datasets_list[test_dataset])
    test_dataset_list.append(counter_test_datasets_list[test_dataset])

    # merge dataset
    merged_train_df = pd.concat(train_dataset_list, ignore_index=True)
    merged_test_df = pd.concat(test_dataset_list, ignore_index=True)

    # create dir or check
    os.makedirs(f"{OUTPUT_BASE_DIR}/{test_circuit}",exist_ok=True)
    # save files
    merged_train_df.to_csv(f"{OUTPUT_BASE_DIR}/{test_circuit}/{TRAIN_DATASET_NAME}", index=False)
    merged_test_df.to_csv(f"{OUTPUT_BASE_DIR}/{test_circuit}/test.csv", index=False)


end_time = time.time()
elapsed = end_time - start_time
print(f"time taken: {elapsed:.4f} seconds")

time taken: 0.3424 seconds


# MOVING IMAGEs

In [18]:
import shutil
import time

In [20]:
start_time = time.time()

total_images = 0
for circuit in CIRCUIT_LIST:
    # make dir
    os.makedirs(os.path.join(OUTPUT_BASE_DIR,circuit,'train'),exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_BASE_DIR,circuit,'test'),exist_ok=True)
    
    print(f"moving clockwise {circuit} train datasets...")
    # moving train datasets
    # clockwise moving
    image_count = 0
    src_dir = os.path.join(CLOCK_RAW_BASE_DIR,circuit,'train')
    for filename in os.listdir(src_dir):
        if filename.lower().endswith(('.png')):
            image_count += 1
            full_file_name = os.path.join(src_dir, filename)
            shutil.copy(full_file_name, os.path.join(OUTPUT_BASE_DIR,circuit,'train'))

    total_images += image_count
    print(f"completed clockwise train {circuit} : {image_count} images")
    
    ## moving counter clock wise train dataset
    print(f"moving counter clockwise {circuit} train datasets...")
    # moving train datasets
    # clockwise moving
    image_count = 0
    src_dir = os.path.join(COUNTER_RAW_BASE_DIR,circuit,'train')
    for filename in os.listdir(src_dir):
        if filename.lower().endswith(('.png')):
            image_count += 1
            full_file_name = os.path.join(src_dir, filename)
            shutil.copy(full_file_name, os.path.join(OUTPUT_BASE_DIR,circuit,'train'))

    total_images += image_count
    print(f"completed clockwise train {circuit} : {image_count} images")
    
    
    
    # moving test datasets
    print(f"moving clockwise {circuit} test datasets...")
    # moving test datasets
    # clockwise moving
    image_count = 0
    src_dir = os.path.join(CLOCK_RAW_BASE_DIR,circuit,'test')
    for filename in os.listdir(src_dir):
        if filename.lower().endswith(('.png')):
            image_count += 1
            full_file_name = os.path.join(src_dir, filename)
            shutil.copy(full_file_name, os.path.join(OUTPUT_BASE_DIR,circuit,'test'))

    total_images += image_count
    print(f"completed clockwise test {circuit} : {image_count} images")
    
    ## moving counter clock wise test dataset
    print(f"moving counter clockwise {circuit} test datasets...")
    # moving test datasets
    # clockwise moving
    image_count = 0
    src_dir = os.path.join(COUNTER_RAW_BASE_DIR,circuit,'test')
    for filename in os.listdir(src_dir):
        if filename.lower().endswith(('.png')):
            image_count += 1
            full_file_name = os.path.join(src_dir, filename)
            shutil.copy(full_file_name, os.path.join(OUTPUT_BASE_DIR,circuit,'test'))

    total_images += image_count
    print(f"completed clockwise test {circuit} : {image_count} images")

elapsed_time = time.time() - start_time
print(f"total images: {total_images}")
print(f"total time taken: {elapsed_time:.4f} seconds")

moving clockwise simple_circuit train datasets...
completed clockwise train simple_circuit : 33751 images
moving counter clockwise simple_circuit train datasets...
completed clockwise train simple_circuit : 29035 images
moving clockwise simple_circuit test datasets...
completed clockwise test simple_circuit : 4816 images
moving counter clockwise simple_circuit test datasets...
completed clockwise test simple_circuit : 4939 images
moving clockwise montreal_circuit train datasets...
completed clockwise train montreal_circuit : 23433 images
moving counter clockwise montreal_circuit train datasets...
completed clockwise train montreal_circuit : 20243 images
moving clockwise montreal_circuit test datasets...
completed clockwise test montreal_circuit : 15134 images
moving counter clockwise montreal_circuit test datasets...
completed clockwise test montreal_circuit : 13731 images
moving clockwise montmelo_circuit train datasets...
completed clockwise train montmelo_circuit : 28354 images
movi